In [1]:
!pip install -q scikit-fuzzy
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.feature_selection import mutual_info_classif

np.random.seed(42)
data = load_wine(as_frame=True)
X, y = data.data, data.target
print(X.shape, y.value_counts().to_dict())
X.describe().T[["min", "mean", "max"]]

(178, 13) {1: 71, 0: 59, 2: 48}


                                 min        mean      max
alcohol                        11.03   13.000618    14.83
malic_acid                      0.74    2.336348     5.80
ash                             1.36    2.366517     3.23
alcalinity_of_ash              10.60   19.494944    30.00
magnesium                      70.00   99.741573   162.00
total_phenols                   0.98    2.295112     3.88
flavanoids                      0.34    2.029270     5.08
nonflavanoid_phenols            0.13    0.361854     0.66
proanthocyanins                 0.41    1.590899     3.58
color_intensity                 1.28    5.058090    13.00
hue                             0.48    0.957449     1.71
od280/od315_of_diluted_wines    1.27    2.611685     4.00
proline                       278.00  746.893258  1680.00

In [3]:
top = ["flavanoids", "proline", "color_intensity",
       "od280/od315_of_diluted_wines", "alcohol", "hue", "total_phenols"]
corr = X[top].corr().round(2)
print(corr)

import seaborn as sns
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Aday özelliklerin korelasyonu")
plt.tight_layout(); plt.show()

                              flavanoids  proline  color_intensity  \
flavanoids                          1.00     0.49            -0.17   
proline                             0.49     1.00             0.32   
color_intensity                    -0.17     0.32             1.00   
od280/od315_of_diluted_wines        0.79     0.31            -0.43   
alcohol                             0.24     0.64             0.55   
hue                                 0.54     0.24            -0.52   
total_phenols                       0.86     0.50            -0.06   

                              od280/od315_of_diluted_wines  alcohol   hue  \
flavanoids                                            0.79     0.24  0.54   
proline                                               0.31     0.64  0.24   
color_intensity                                      -0.43     0.55 -0.52   
od280/od315_of_diluted_wines                          1.00     0.07  0.57   
alcohol                                               

<Figure size 700x500 with 2 Axes>

In [4]:
from sklearn.model_selection import cross_val_score, RepeatedStratifiedKFold
from sklearn.tree import DecisionTreeClassifier

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
adaylar = {
    "A: flav+prol+color+alcohol": ["flavanoids", "proline", "color_intensity", "alcohol"],
    "B: flav+prol+color+hue":     ["flavanoids", "proline", "color_intensity", "hue"],
    "C: flav+prol+color+od280":   ["flavanoids", "proline", "color_intensity", "od280/od315_of_diluted_wines"],
}
for ad, cols in adaylar.items():
    s = cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=42), X[cols], y, cv=cv)
    print(f"{ad}: {s.mean():.3f} ± {s.std():.3f}")

A: flav+prol+color+alcohol: 0.931 ± 0.043
B: flav+prol+color+hue: 0.931 ± 0.048
C: flav+prol+color+od280: 0.912 ± 0.048


In [2]:
mi = pd.Series(mutual_info_classif(X, y, random_state=42), index=X.columns)
print(mi.sort_values(ascending=False).round(3))

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, col in zip(axes.ravel(), mi.sort_values(ascending=False).index[:6]):
    for c in sorted(y.unique()):
        ax.hist(X.loc[y == c, col], alpha=0.5, label=f"sınıf {c}")
    ax.set_title(col)
axes[0, 0].legend()
plt.tight_layout(); plt.show()

flavanoids                      0.668
proline                         0.577
color_intensity                 0.548
od280/od315_of_diluted_wines    0.516
alcohol                         0.474
hue                             0.466
total_phenols                   0.402
proanthocyanins                 0.305
malic_acid                      0.282
alcalinity_of_ash               0.238
magnesium                       0.213
nonflavanoid_phenols            0.121
ash                             0.079
dtype: float64


<Figure size 1200x600 with 6 Axes>